In [2]:
# Установка зависимостей
!pip install aiogram openai pandas numpy tiktoken nest_asyncio python-dotenv

import nest_asyncio
nest_asyncio.apply()

import asyncio
import logging
import os
import pandas as pd
import numpy as np
from typing import List, Tuple
import tiktoken
from openai import OpenAI
from aiogram import Bot, Dispatcher, types, F
from aiogram.filters import Command
from aiogram.types import Message, InlineKeyboardMarkup, InlineKeyboardButton
from aiogram.utils.keyboard import InlineKeyboardBuilder
from getpass import getpass

# Настройка логирования
logging.basicConfig(level=logging.INFO)
print("Библиотеки установлены и")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.7/751.7 kB 16.7 MB/s eta 0:00:00
Библиотеки установлены и


In [3]:
# Ввод API ключа и токена бота
BOT_TOKEN = getpass("Токен Telegram бота (забрать у @BotFather): ")
OPENAI_API_KEY = getpass("OpenAI API Key (забрать у openai.com): ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Инициализация OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

print("Ключи сохранены")

Токен Telegram бота (забрать у @BotFather): ··········
OpenAI API Key (забрать у openai.com): ··········
Ключи сохранены


In [4]:
# Создание базы знаний
# Путь к файлу базы знаний (можно загрузить в Colab)
CSV_PATH = "./knowledge_base.csv"

# Функция для загрузки базы знаний
def load_knowledge_base():
    """Загружает базу знаний из CSV или создает тестовую"""

    try:
        df = pd.read_csv(CSV_PATH)
        print(f"Загружена база знаний из {CSV_PATH}")

        # Преобразуем эмбединги из строки в список, если нужно
        if 'embedding' in df.columns and isinstance(df['embedding'].iloc[0], str):
            df['embedding'] = df['embedding'].apply(lambda x: np.array(eval(x)))

        return df
    except FileNotFoundError:
        print("Файл базы знаний не найден. Создаем тестовую базу...")

        # Тестовая база знаний
        test_data = {
            'text': [
                "Python - это высокоуровневый язык программирования, созданный Гвидо ван Россумом в 1991 году. Он известен своей простотой и читаемостью кода.",
                "Telegram боты - это автоматизированные аккаунты, которыми управляют программы через Telegram Bot API. Они могут отвечать на сообщения, отправлять уведомления и многое другое.",
                "Yandex Cloud - это облачная платформа от Яндекса, предоставляющая сервисы для разработки и развертывания приложений: Cloud Functions, API Gateway, YDB, Object Storage.",
                "Искусственный интеллект (ИИ) - это область компьютерных наук, занимающаяся созданием систем, способных выполнять задачи, требующие человеческого интеллекта.",
                "Машинное обучение - это подполе ИИ, позволяющее системам обучаться на данных без явного программирования правил.",
                "Нейронные сети - это вычислительные системы, вдохновленные биологическими нейронными сетями, способные распознавать образы и паттерны.",
                "RAG (Retrieval-Augmented Generation) - это техника, комбинирующая поиск релевантного контекста с генерацией ответов LLM.",
                "OpenAI API предоставляет доступ к моделям GPT для генерации текста и моделям эмбедингов для поиска.",
                "Telegram Bot API позволяет создавать ботов с клавиатурами, инлайн-кнопками и обработкой различных типов сообщений.",
                "Векторные базы данных (как Pinecone, Weaviate, Qdrant) оптимизированы для хранения и поиска эмбедингов."
            ]
        }
        df = pd.DataFrame(test_data)

        print(f"Создана тестовая база с {len(df)} записями")
        print("Для использования своей базы: загрузите CSV файл и измените CSV_PATH")

        return df

# Загружаем базу знаний
knowledge_base = load_knowledge_base()

# Информация о базе знаний
KB_INFO = {
    "topic": "Программирование, ИИ и облачные технологии",
    "records_count": len(knowledge_base),
    "example_query": "Что такое Python?",
    "description": "База знаний содержит информацию о языках программирования, ИИ, облачных сервисах и технологиях."
}

print(f"\nИнформация о базе знаний:")
print(f"   Тематика: {KB_INFO['topic']}")
print(f"   Записей: {KB_INFO['records_count']}")
print(f"   Пример запроса: {KB_INFO['example_query']}")

Файл базы знаний не найден. Создаем тестовую базу...
Создана тестовая база с 10 записями
Для использования своей базы: загрузите CSV файл и измените CSV_PATH

Информация о базе знаний:
   Тематика: Программирование, ИИ и облачные технологии
   Записей: 10
   Пример запроса: Что такое Python?


In [5]:
# Функции для эмбедингов и поиска
# Конфигурация
EMBEDDING_MODEL = "text-embedding-ada-002"
GPT_MODEL = "gpt-3.5-turbo"

# Функция получения эмбединга
def get_embedding(text: str) -> List[float]:
    """Получает эмбединг для текста через OpenAI API"""
    try:
        response = client.embeddings.create(
            input=[text],
            model=EMBEDDING_MODEL
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"Ошибка получения эмбединга: {e}")
        return None

# Косинусное сходство
def cosine_similarity(a, b):
    """Вычисляет косинусное сходство между двумя векторами"""
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Поиск релевантного контекста
def find_relevant_context(query: str, top_k: int = 3) -> List[Tuple[str, float]]:
    """Находит наиболее релевантные фрагменты из базы знаний"""
    # Если в базе нет эмбедингов, генерируем их на лету
    if 'embedding' not in knowledge_base.columns:
        print("Генерация эмбедингов для базы знаний...")
        knowledge_base['embedding'] = knowledge_base['text'].apply(lambda x: get_embedding(x))

    # Получаем эмбединг запроса
    query_embedding = get_embedding(query)
    if query_embedding is None:
        return []

    # Вычисляем сходство со всеми записями
    similarities = []
    for idx, row in knowledge_base.iterrows():
        doc_embedding = row['embedding']
        if doc_embedding is not None:
            similarity = cosine_similarity(query_embedding, doc_embedding)
            similarities.append((row['text'], similarity))

    # Сортируем по убыванию сходства
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

# Генерация ответа на основе контекста
def generate_answer(query: str, context: List[Tuple[str, float]]) -> str:
    """Генерирует ответ на основе найденного контекста"""
    if not context:
        return "Извините, я не нашел релевантной информации в базе знаний для ответа на ваш вопрос."

    # Формируем контекст
    context_text = "\n\n---\n\n".join([f"Источник {i+1}:\n{text}" for i, (text, _) in enumerate(context)])

    # Промпт для GPT
    prompt = f"""Ты - полезный помощник, который отвечает на вопросы, используя ТОЛЬКО предоставленный контекст.

ПРАВИЛА:
1. Отвечай кратко и по существу
2. Если ответа нет в контексте, скажи: "Информация не найдена в базе знаний"
3. Не добавляй информацию из своего знания - только из контекста

КОНТЕКСТ:
{context_text}

ВОПРОС: {query}

ОТВЕТ:"""

    try:
        response = client.chat.completions.create(
            model=GPT_MODEL,
            messages=[
                {"role": "system", "content": "Ты полезный ассистент. Отвечай только на основе предоставленного контекста."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.5,
            max_tokens=500
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Ошибка при генерации ответа: {e}"

print("Функции готовы")

Функции готовы


In [6]:
# Создаем бота
bot = Bot(token=BOT_TOKEN)
dp = Dispatcher()

# Словарь для истории
user_history = {}

# Обработчик /start
@dp.message(Command("start"))
async def cmd_start(message: Message):
    welcome_text = (
        "*Привет! Я бот с базой знаний!*\n\n"
        "Я умею отвечать на вопросы, используя информацию из моей базы знаний.\n\n"
        "*Доступные команды:*\n"
        "/help - информация о базе знаний\n"
        "/about - обо мне\n\n"
        "Просто задай свой вопрос, и я постараюсь на него ответить!"
    )
    await message.answer(welcome_text, parse_mode="Markdown")

# Обработчик /help
@dp.message(Command("help"))
async def cmd_help(message: Message):
    help_text = (
        "*Информация о базе знаний*\n\n"
        f"*Тематика:* {KB_INFO['topic']}\n"
        f"*Количество записей:* {KB_INFO['records_count']}\n"
        f"*Пример запроса:* `{KB_INFO['example_query']}`\n\n"
        f"*Описание:* {KB_INFO['description']}\n\n"
        "*Как это работает:*\n"
        "1. Я нахожу в базе знаний релевантные фрагменты\n"
        "2. Использую их как контекст для ChatGPT\n"
        "3. Генерирую ответ на основе контекста\n\n"
        "Попробуй задать вопрос по теме базы знаний!"
    )
    await message.answer(help_text, parse_mode="Markdown")

# Обработчик /about
@dp.message(Command("about"))
async def cmd_about(message: Message):
    about_text = (
        "*О боте*\n\n"
        "Этот бот создан в учебных целях для демонстрации технологии RAG "
        "(Retrieval-Augmented Generation).\n\n"
        "*Технологии:*\n"
        "aiogram3 - библиотека для Telegram ботов\n"
        "OpenAI API - эмбединги и генерация ответов\n"
        "Cosine similarity - поиск релевантного контекста\n\n"
        "*Совет:* Задавайте вопросы, близкие по теме к базе знаний!"
    )
    await message.answer(about_text, parse_mode="Markdown")

# Обработчик текстовых сообщений
@dp.message(F.text)
async def answer_question(message: Message):
    user_id = message.from_user.id
    query = message.text.strip()

    # Показываем индикатор
    await bot.send_chat_action(chat_id=user_id, action="typing")

    # Отправляем сообщение о поиске
    thinking_msg = await message.answer("🔍 Ищу ответ в базе знаний...")

    try:
        # Поиск релевантного контекста
        relevant_context = find_relevant_context(query)

        if not relevant_context:
            await thinking_msg.edit_text(
                "Не удалось найти информацию в базе знаний.\n\n"
                "Попробуйте переформулировать вопрос или задать вопрос по теме:\n"
                f"`{KB_INFO['topic']}`",
                parse_mode="Markdown"
            )
            return

        # Генерация ответа
        answer = generate_answer(query, relevant_context)

        # Формируем ответ с источниками
        sources_text = "\n\n*Источники:*\n"
        for i, (text, score) in enumerate(relevant_context[:2], 1):
            preview = text[:80] + "..." if len(text) > 80 else text
            sources_text += f"{i}. {preview}\n"

        # Клавиатура для обратной связи
        keyboard = InlineKeyboardBuilder()
        keyboard.add(InlineKeyboardButton(text="Полезно", callback_data="feedback_positive"))
        keyboard.add(InlineKeyboardButton(text="Бесполезно", callback_data="feedback_negative"))

        # Отправляем ответ
        await thinking_msg.edit_text(
            f"💬 *Ответ:*\n{answer}\n{sources_text}",
            parse_mode="Markdown",
            reply_markup=keyboard.as_markup()
        )

    except Exception as e:
        await thinking_msg.edit_text(f"Произошла ошибка: {str(e)}\nПожалуйста, попробуйте позже.")
        print(f"Ошибка: {e}")

# Обработчик обратной связи
@dp.callback_query(F.data.startswith("feedback_"))
async def handle_feedback(callback: types.CallbackQuery):
    if callback.data == "feedback_positive":
        await callback.answer("Спасибо за отзыв!", show_alert=False)
    else:
        await callback.answer("Спасибо за обратную связь!", show_alert=False)

    # Удаляем кнопки
    await callback.message.edit_reply_markup(reply_markup=None)

# Обработчик ошибок
@dp.errors()
async def error_handler(update, exception):
    print(f"Unhandled exception: {exception}")
    return True

print("Бот создан!")

Бот создан!


In [7]:
# Запуск бота
async def main():
    print("Запуск бота...")
    print(f"База знаний: {KB_INFO['records_count']} записей")
    print(f"Тематика: {KB_INFO['topic']}")
    print("Бот готов к работе! Напишите ему в Telegram")
    print("")
    print("Доступные команды в Telegram:")
    print("/start - начать")
    print("/help - информация о базе знаний")
    print("/about - о боте")

    await dp.start_polling(bot)

# Запуск
if __name__ == "__main__":
    try:
        asyncio.run(main())
    except RuntimeError:

        loop = asyncio.get_event_loop()
        loop.create_task(main())
        print("Бот запущен в фоновом режиме")

        # Активная ячейка
        import time
        try:
            while True:
                time.sleep(1)
        except KeyboardInterrupt:
            print("Бот остановлен")

Запуск бота...
База знаний: 10 записей
Тематика: Программирование, ИИ и облачные технологии
Бот готов к работе! Напишите ему в Telegram

Доступные команды в Telegram:
/start - начать
/help - информация о базе знаний
/about - о боте
